# p0 — Population Statistics

**Purpose (optional notebook):** Characterize the population of models in `data_raw/models.parquet` before any IRT fitting. This notebook does not produce outputs consumed by later pipeline steps; it exists to verify that the cohort and architecture distributions are reasonable.

**What this notebook covers:**
1. Cohort descriptive statistics — model counts, parameter size distributions, and KS tests comparing each cohort to the earliest reference cohort
2. Architecture distribution — how the mix of architectures (Llama, Qwen2, Mistral, Gemma2, …) varies across cohorts
3. Per-architecture descriptive statistics — parameter size breakdown for the top-10 architectures

**Key question:** Are the cohorts compositionally stable enough for IRT parameter comparisons to be meaningful, or do shifts in model size / architecture confound temporal trends?

In [3]:
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp

## 1. Load Model Metadata

We retain four columns:
- `fullname` — HuggingFace model identifier
- `Architecture` — transformer class name (e.g. `LlamaForCausalLM`)
- `#Params (B)` — parameter count in billions
- `cohort` — quarterly release window the model was first evaluated in (Q2-2023 … Q1-2025)

In [4]:
df = pd.read_parquet("../data_raw/models.parquet")[
    ["fullname", "Architecture", "#Params (B)", "cohort"]
]

print(f"{len(df):,} models loaded")
print(df.dtypes)
df.head()

KeyError: "['cohort'] not in index"

## 2. Architecture Overview

Quick frequency table to see which architecture classes dominate the leaderboard.
We later focus our IRT analysis on the four most common families: **Llama, Qwen2, Mistral, Gemma2**.

In [ ]:
df["Architecture"].value_counts()

## 3. Cohort Descriptive Statistics (Table 1)

For each quarterly cohort we compute:
- **N** — number of models
- **mean / SD of #Params (B)** — typical model size
- **Size breakdown** — % of models in three bins: `<7B`, `7–70B`, `>70B`
- **KS statistic & p-value** — two-sample Kolmogorov–Smirnov test of parameter-size distribution vs. the earliest cohort (Q2-2023)

A significant KS p-value (< 0.05) indicates the cohort's size distribution has shifted. This is expected as the field matures, but large shifts may confound IRT comparisons — we note any flagged cohorts in the paper.

In [ ]:
def cohort_sort_key(cohort):
    """Sort cohort strings (e.g. 'Q3-2024') chronologically."""
    if cohort == "Unknown":
        return (9999, 9)
    q, year = cohort.split("-")
    return (int(year), int(q[1]))


def size_bin(params):
    """Bin parameter count into three standard size categories."""
    if pd.isna(params): return "unknown"
    if params < 7:      return "<7B"
    if params <= 70:    return "7-70B"
    return ">70B"


df["size_bin"] = df["#Params (B)"].apply(size_bin)

cohorts   = sorted(df["cohort"].dropna().unique(), key=cohort_sort_key)
ref       = cohorts[0]
ref_params = df[df["cohort"] == ref]["#Params (B)"].dropna()

rows = []
for cohort in cohorts:
    sub    = df[df["cohort"] == cohort]
    params = sub["#Params (B)"].dropna()
    n      = len(sub)

    ks_stat, ks_p = (
        (np.nan, np.nan) if cohort == ref else ks_2samp(ref_params, params)
    )

    rows.append({
        "cohort":    cohort,
        "N":         n,
        "mean_B":    params.mean().round(1),
        "sd_B":      params.std().round(1),
        "pct_<7B":   round((sub["size_bin"] == "<7B").sum()   / n * 100, 1),
        "pct_7-70B": round((sub["size_bin"] == "7-70B").sum() / n * 100, 1),
        "pct_>70B":  round((sub["size_bin"] == ">70B").sum()  / n * 100, 1),
        "KS_stat":   round(ks_stat, 3) if not np.isnan(ks_stat) else "ref",
        "p_value":   round(ks_p,    4) if not np.isnan(ks_p)    else "ref",
    })

table1 = pd.DataFrame(rows).sort_values(
    "cohort", key=lambda s: s.map(cohort_sort_key)
)
print("Table 1: Cohort descriptive statistics")
print(table1.to_string(index=False))

## 4. Architecture Distribution Across Cohorts

Here we check whether the *architecture mix* is stable across cohorts. We rank architectures by global frequency and apply a KS test on those ranks between each cohort and the reference.

A KS p-value near 1.0 means the cohort has essentially the same architecture distribution as the reference — Llama-dominated throughout, which is consistent with leaderboard trends.

In [ ]:
arch_rank    = df["Architecture"].value_counts().rank(ascending=False)
df["arch_rank"] = df["Architecture"].map(arch_rank)
ref_arch_ranks  = df[df["cohort"] == ref]["arch_rank"].dropna()

rows_arch = []
for cohort in cohorts:
    sub        = df[df["cohort"] == cohort]
    sub_ranks  = sub["arch_rank"].dropna()
    top3       = sub["Architecture"].value_counts().head(3).index.tolist()

    ks_stat, ks_p = (
        (np.nan, np.nan) if cohort == ref else ks_2samp(ref_arch_ranks, sub_ranks)
    )

    rows_arch.append({
        "cohort":   cohort,
        "N":        len(sub),
        "top_3":    ", ".join(top3),
        "KS_stat":  round(ks_stat, 3) if not np.isnan(ks_stat) else "ref",
        "p_value":  round(ks_p,    4) if not np.isnan(ks_p)    else "ref",
    })

print("Architecture distribution KS analysis")
print(pd.DataFrame(rows_arch).to_string(index=False))

## 5. Per-Architecture Descriptive Statistics (Table 2)

Same size-bin breakdown as Table 1, but grouped by architecture family instead of cohort. Reference architecture is `LlamaForCausalLM` (the most common). The KS test compares each architecture's parameter-size distribution to Llama's.

In [ ]:
top_archs      = df["Architecture"].value_counts().head(10).index
ref_arch_name  = top_archs[0]
ref_params_arch = df[df["Architecture"] == ref_arch_name]["#Params (B)"].dropna()

rows_a = []
for arch in top_archs:
    sub    = df[df["Architecture"] == arch]
    params = sub["#Params (B)"].dropna()
    n      = len(sub)

    ks_stat, ks_p = (
        (np.nan, np.nan) if arch == ref_arch_name else ks_2samp(ref_params_arch, params)
    )

    rows_a.append({
        "architecture": arch,
        "N":            n,
        "mean_B":       params.mean().round(1),
        "sd_B":         params.std().round(1),
        "pct_<7B":      round((sub["size_bin"] == "<7B").sum()   / n * 100, 1),
        "pct_7-70B":    round((sub["size_bin"] == "7-70B").sum() / n * 100, 1),
        "pct_>70B":     round((sub["size_bin"] == ">70B").sum()  / n * 100, 1),
        "KS_stat":      round(ks_stat, 3) if not np.isnan(ks_stat) else "ref",
        "p_value":      round(ks_p,    4) if not np.isnan(ks_p)    else "ref",
    })

table2 = pd.DataFrame(rows_a).sort_values("N", ascending=False)
print("Table 2: Architecture descriptive statistics (top 10)")
print(table2.to_string(index=False))